# 01. LangGraph 与 OpenAI Agents SDK

两大主流框架，代表了两种不同的 Agent 编程范式：

- **LangGraph**：Graph-first。一切是节点+边。适合需要持久化、人机交互、长时间运行的 Agent。
- **OpenAI Agents SDK**：Agent-first。一切是 Agent+Handoff。适合快速构建 OpenAI 生态内的多 Agent 应用。

> **检查点**：能说出 LangGraph 的 StateGraph 三要素（State/Nodes/Edges）和 OpenAI SDK 的三原语（Agents/Handoffs/Guardrails）。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'agents'))

from scripts.nb_helpers import setup, chat, chat_json, chat_with_tools, show_trace
client = setup()


🌐 API 模式 — model=deepseek-flash  base_url=https://api.deepseek.com


In [2]:
# LangGraph 风格：StateGraph with conditional routing
from dataclasses import dataclass
from enum import Enum

class Step(Enum):
    RESEARCH = "research"
    SUMMARIZE = "summarize"
    REVIEW = "review"
    DONE = "done"

@dataclass
class State:
    goal: str; step: Step = Step.RESEARCH
    content: str = ""; approved: bool = False

def node_research(state: State) -> State:
    state.content = f"Research on '{state.goal}': found 5 sources."
    state.step = Step.SUMMARIZE
    return state

def node_summarize(state: State) -> State:
    state.content += f"\nSummary: key insights from research."
    state.step = Step.REVIEW
    return state

def node_review(state: State) -> State:
    state.approved = len(state.content) > 50
    state.step = Step.DONE
    return state

def route(state: State):
    return {Step.RESEARCH: node_research, Step.SUMMARIZE: node_summarize,
            Step.REVIEW: node_review}[state.step]

# 执行
state = State(goal="AI trends 2025")
while state.step != Step.DONE:
    prev = state.step
    state = route(state)(state)
    print(f"  {prev.value} → {state.step.value}")
print(f"\nApproved: {state.approved}, Content length: {len(state.content)}")

  research → summarize
  summarize → review
  review → done

Approved: True, Content length: 83


In [3]:
# OpenAI SDK 风格：Agent + Handoff + Guardrails
@dataclass
class Agent:
    name: str; instructions: str; handoffs: list = None
    def run(self, input_text: str, history: list = None) -> dict:
        return {"agent": self.name, "output": f"[{self.name}] {input_text}", "next": None}

class Guardrail:
    @staticmethod
    def check_input(text: str) -> bool:
        blocked = ["hack", "exploit", "bypass"]
        return not any(w in text.lower() for w in blocked)
    @staticmethod
    def check_output(text: str) -> bool:
        return "error" not in text.lower()

# 构建 Agent 网络
researcher = Agent("researcher", "Find and verify facts")
writer = Agent("writer", "Synthesize into clear text")
reviewer = Agent("reviewer", "Check accuracy and completeness")

# 运行
for agent in [researcher, writer, reviewer]:
    if not Guardrail.check_input("market trends 2025"):
        print(f"BLOCKED at {agent.name}"); break
    result = agent.run("market trends 2025")
    print(f"{agent.name}: {result['output'][:60]}...")

researcher: [researcher] market trends 2025...
writer: [writer] market trends 2025...
reviewer: [reviewer] market trends 2025...


## 选择指南

| 需求 | 推荐 |
|------|------|
| 需要持久化、长时间运行、崩溃恢复 | LangGraph |
| OpenAI 生态内快速构建、需要 built-in tracing | OpenAI Agents SDK |
| 需要精确控制每一步的状态流转 | LangGraph |
| 需要最简单的 Agent 间 delegation | OpenAI Agents SDK |

> **检查点**：能为你的具体需求在 LangGraph 和 OpenAI SDK 之间做出有理由的选择。

## 真实 API 实验

`sdk-live` 风格：真实 LLM 的 instructions → system prompt、handoff → 输出传递（OpenAI Agents SDK 的核心机制）。


In [4]:
SCRIPT = ROOT / 'agents' / 'orchestration' / 'scripts' / 'framework_styles_demo.py'
import subprocess, sys
result = subprocess.run([sys.executable, str(SCRIPT), '--style', 'sdk-live', '--use-api', '--verbose',
                         '--scenario', 'agent 记忆系统'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr, file=sys.stderr)


Style: sdk-live, Scenario: agent 记忆系统
  [sdk-live:researcher] - 主流分层：短期/工作记忆保存当前对话与任务状态；长期记忆常分为情景记忆（具体交互）、语义记忆（事实知识）、程序记忆（技能/策略），多用向量数据库、知识图谱或分层缓存持久化。
- 核心机制：写入（抽...
  [sdk-live:writer] Memory in LLM systems is typically layered into short-term/working memory for current dialogue and t...
  [sdk-live:reviewer] APPROVED — The summary accurately captures the standard layered LLM memory taxonomy, pipeline, context-injection role, challenges, and evaluation metrics without material factual errors.

SDK-live output: 3 real LLM agents completed — reviewer said: APPROVED — The summary accurately captures the standard laye



## 练习

1. 在 LangGraph 风格中增加 conditional edge：如果 review 不通过，路由回 research 重新开始。
2. 在 SDK 风格中实现 Handoff：researcher 完成后自动将结果 handoff 给 writer。
3. 两个框架都支持 streaming。设计一个 streaming agent 的接口，使两个框架都能实现。